In [1]:
#imports
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [2]:
# The code below was used to import the TLE data from the API and turn it into a CSV file. If you wish to use the most recent data from the API,
# feel free to create an account on spacetrack.org and imput your email and password into this code. Otherwise, please use the provided 
# leoSatellites.csv file. This code is commented out as it will not work without valid credentials.
""" import json
import pandas as pd
from spacetrack import SpaceTrackClient
import spacetrack.operators as op

st = SpaceTrackClient(identity='address@gmail.com', password='XXXXXXXXXXXX.')

# Fetch the data, null-val decay rate means we are only analysing currently orbiting satellites
leo_raw = st.gp(period=op.inclusive_range(84, 128), decay_date='null-val', format='json')

try:
    # Attempt to parse the data
    data_list = json.loads(leo_raw)
    df = pd.DataFrame(data_list)
    
    # Check if we actually got data
    if not df.empty:
        print("Success! Top 5 Owners:")
        print(df['COUNTRY_CODE'].value_counts().head())
        df.to_csv("leoSatellites.csv", index=False)
    else:
        print("Query successful, but no satellites found in that range.")

except json.JSONDecodeError:
    # If the API returned a text error message instead of JSON data
    print("API Error or Rate Limit Hit. The server said:")
    print(leo_raw) """

' import json\nimport pandas as pd\nfrom spacetrack import SpaceTrackClient\nimport spacetrack.operators as op\n\nst = SpaceTrackClient(identity=\'address@gmail.com\', password=\'XXXXXXXXXXXX.\')\n\n# Fetch the data, null-val decay rate means we are only analysing currently orbiting satellites\nleo_raw = st.gp(period=op.inclusive_range(84, 128), decay_date=\'null-val\', format=\'json\')\n\ntry:\n    # Attempt to parse the data\n    data_list = json.loads(leo_raw)\n    df = pd.DataFrame(data_list)\n    \n    # Check if we actually got data\n    if not df.empty:\n        print("Success! Top 5 Owners:")\n        print(df[\'COUNTRY_CODE\'].value_counts().head())\n        df.to_csv("leoSatellites.csv", index=False)\n    else:\n        print("Query successful, but no satellites found in that range.")\n\nexcept json.JSONDecodeError:\n    # If the API returned a text error message instead of JSON data\n    print("API Error or Rate Limit Hit. The server said:")\n    print(leo_raw) '

## I. Problem Definition: Satellite Orbit Classification
In this project, we utilize **Two-Line Element (TLE)** data to distinguish between active payloads and space debris. As the number of tracked objects in Earth's orbit increases, automated classification becomes vital for space situational awareness. 

**Goal:** Implement a **Random Forest Classifier** to predict the `OBJECT_TYPE` based on orbital parameters like eccentricity, apoapsis, and ballistic coefficients.

## II. Data Selection, Investigation, and Preprocessing

In [ ]:
# Load Dataset csv file

# Contains Payload, Debris, Unknown, TBA, and Rocket Body
df = pd.read_csv('CSV Data Sources/leoSatellites.csv')

# Preprocessing removal of identifiers and non-numeric metadata that do not contribute to physical orbital characteristics
df = df.drop(columns=['NORAD_CAT_ID', 'TLE_LINE0', 'GP_ID', 'FILE', 'OBJECT_ID', 'OBJECT_NAME', 'CREATION_DATE'])

# Only Contains Payload and Debris
df_reduced = df[df['OBJECT_TYPE'].isin(['DEBRIS', 'PAYLOAD'])]

# Data investigation
print(f'Unique Satelite Categories are: {df["OBJECT_TYPE"].unique()} \n')
print(df['OBJECT_TYPE'].value_counts())
df.head()

Unique Satelite Categories are: ['PAYLOAD' 'ROCKET BODY' 'DEBRIS' 'UNKNOWN' 'TBA'] 

OBJECT_TYPE
PAYLOAD        15339
DEBRIS         10156
UNKNOWN         1400
ROCKET BODY     1010
TBA              130
Name: count, dtype: int64


,CCSDS_OMM_VERS,COMMENT,ORIGINATOR,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,EPOCH,MEAN_MOTION,ECCENTRICITY,...,APOAPSIS,PERIAPSIS,OBJECT_TYPE,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,TLE_LINE1,TLE_LINE2
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,18 SPCS,EARTH,TEME,UTC,SGP4,2026-04-13T07:01:10.958880,11.903476,0.144432,...,2894.631,554.108,PAYLOAD,MEDIUM,US,1959-02-17,AFETR,NaN,1 00011U 59001A 26103.29248795 .00000796 0...,2 00011 32.8673 330.7473 1444321 47.5661 323...
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,18 SPCS,EARTH,TEME,UTC,SGP4,2026-04-13T07:07:55.894080,11.485950,0.164627,...,3285.577,553.534,ROCKET BODY,MEDIUM,US,1959-02-17,AFETR,NaN,1 00012U 59001B 26103.29717470 .00000211 0...,2 00012 32.8962 93.9114 1646267 121.5399 255...
2,3.0,GENERATED VIA SPACE-TRACK.ORG API,18 SPCS,EARTH,TEME,UTC,SGP4,2026-04-13T04:48:54.741312,11.619216,0.163730,...,3204.163,507.811,PAYLOAD,MEDIUM,US,1959-09-18,AFETR,NaN,1 00020U 59007A 26103.20063358 .00000386 0...,2 00020 33.3387 140.9176 1637304 65.1838 310...
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,18 SPCS,EARTH,TEME,UTC,SGP4,2026-04-13T06:37:05.188800,15.276230,0.007977,...,537.654,428.191,PAYLOAD,MEDIUM,US,1959-10-13,AFETR,NaN,1 00022U 59009A 26103.27575450 .00006199 0...,2 00022 50.2441 60.7846 0079771 222.8855 136...
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,18 SPCS,EARTH,TEME,UTC,SGP4,2026-04-12T22:15:17.030016,14.786947,0.002230,...,649.084,617.814,PAYLOAD,MEDIUM,US,1960-04-01,AFETR,NaN,1 29U 60002B 26102.92728044 .00000271 0...,2 29 48.3796 334.7736 0022299 346.5924 13...


## III. Data Mining Methods
- Define X (feature matrix) and y (target vector)
- Perform the train and test split
- Run the Random Forest Classifiers, with a balanced hyperparameter due to imbalanced dataset

In [4]:
# Since the Random Forest Algorithm only takes numerical values for X, we assign X as the database parameters that are numeric.
# Fruthermore, all of our parameters of interest are numeric, so this choice makes sense.
# Had they been non-numeric as well, would have had to one-hot encode.

X = df.select_dtypes(include=['number'])
print(X)
y = df['OBJECT_TYPE']
print(y)

# Only Debris and Payload
X_reduced = df_reduced.select_dtypes(include=['number'])
y_reduced = df_reduced['OBJECT_TYPE']
print(y_reduced)

       CCSDS_OMM_VERS  MEAN_MOTION  ECCENTRICITY  INCLINATION  RA_OF_ASC_NODE  \
0                 3.0    11.903476      0.144432      32.8673        330.7473   
1                 3.0    11.485950      0.164627      32.8962         93.9114   
2                 3.0    11.619216      0.163730      33.3387        140.9176   
3                 3.0    15.276230      0.007977      50.2441         60.7846   
4                 3.0    14.786947      0.002230      48.3796        334.7736   
...               ...          ...           ...          ...             ...   
28030             3.0    14.715176      0.014306      88.7449        256.8085   
28031             3.0    14.689549      0.012485      88.4851        237.8715   
28032             3.0    14.346675      0.001598      87.8708        197.8746   
28033             3.0    14.135112      0.021626      88.9165        277.4769   
28034             3.0    15.942440      0.002396      87.9628        294.9979   

       ARG_OF_PERICENTER  M

In [5]:
# performs the train test split on all 5 categories
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# performs the train test split on reduced debris and payload categories only
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(X_reduced, y_reduced, test_size=0.2)

# Set up the model with all 5 categories
rnd_clf = RandomForestClassifier(n_estimators=500, class_weight='balanced', n_jobs=-1) # 5 unique categories, class_weight='balanced', removed max leaf nodes

# Set up the reduced
rnd_clf_reduced = RandomForestClassifier(n_estimators=500, n_jobs=-1)

In [6]:
# Fit the general model
rnd_clf.fit(X_train, y_train)

# Fit the reduced model
rnd_clf_reduced.fit(X_train_reduced, y_train_reduced)

,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
# Get the predictions
y_pred = rnd_clf.predict(X_test)
y_pred_reduced = rnd_clf_reduced.predict(X_test_reduced)

## IV. Evaluation and Results

In [8]:
# Get the classification report to display precision, recall, and the f1-score
from sklearn.metrics import classification_report
print(f'The classification model for the general model is:\n{classification_report(y_test, y_pred)}')
print(f'The classification model for the reduced model is:\n{classification_report(y_test_reduced, y_pred_reduced)}')

The classification model for the general model is:
              precision    recall  f1-score   support

      DEBRIS       0.90      0.96      0.93      1995
     PAYLOAD       0.94      0.98      0.96      3106
 ROCKET BODY       0.59      0.31      0.41       195
         TBA       0.86      0.18      0.29        34
     UNKNOWN       0.87      0.40      0.55       277

    accuracy                           0.91      5607
   macro avg       0.83      0.56      0.63      5607
weighted avg       0.91      0.91      0.90      5607

The classification model for the reduced model is:
              precision    recall  f1-score   support

      DEBRIS       0.97      0.96      0.97      2032
     PAYLOAD       0.98      0.98      0.98      3067

    accuracy                           0.97      5099
   macro avg       0.97      0.97      0.97      5099
weighted avg       0.97      0.97      0.97      5099



Reducing the dataset to just payload and debris improved the performance of the model, specifically by improving the precision of the model.

In [9]:
# Which features are the most important?

# General
features = pd.DataFrame(rnd_clf.feature_importances_, index=X.columns)
tree_top_10_features = features.sort_values(by=0, ascending=False).head(10)
print(f'The top ten features for the general model are:\n{tree_top_10_features}')

# Reduced
features_reduced = pd.DataFrame(rnd_clf_reduced.feature_importances_, index=X.columns)
tree_top_10_features_reduced = features.sort_values(by=0, ascending=False).head(10)
print(f'The top ten features for the reduced model are:\n{tree_top_10_features_reduced}')

The top ten features for the general model are:
                        0
ECCENTRICITY     0.144850
BSTAR            0.122585
INCLINATION      0.113908
REV_AT_EPOCH     0.108419
MEAN_MOTION_DOT  0.081164
APOAPSIS         0.070387
PERIAPSIS        0.063374
SEMIMAJOR_AXIS   0.057690
RA_OF_ASC_NODE   0.056008
MEAN_MOTION      0.055960
The top ten features for the reduced model are:
                        0
ECCENTRICITY     0.144850
BSTAR            0.122585
INCLINATION      0.113908
REV_AT_EPOCH     0.108419
MEAN_MOTION_DOT  0.081164
APOAPSIS         0.070387
PERIAPSIS        0.063374
SEMIMAJOR_AXIS   0.057690
RA_OF_ASC_NODE   0.056008
MEAN_MOTION      0.055960


In [10]:
# Does this make sense?
debris = df.loc[df['OBJECT_TYPE'] == 'DEBRIS']
payload = df.loc[df['OBJECT_TYPE'] == 'PAYLOAD']

print(f'Debris Eccentricity is {debris["ECCENTRICITY"].mean()}')
print(f'Payload Eccentricity is {payload["ECCENTRICITY"].mean()}')

print(f'Debris Balistic Coefficient is {debris["BSTAR"].mean()}')
print(f'Payload Balistic Coefficient is {payload["BSTAR"].mean()}')

print(f'Debris Inclination is {debris["INCLINATION"].mean()}')
print(f'Payload Inclination is {payload["INCLINATION"].mean()}')


Debris Eccentricity is 0.013657807914533282
Payload Eccentricity is 0.0012124118436664714
Debris Balistic Coefficient is 0.001847389236886668
Payload Balistic Coefficient is -1.6146042231827337e-05
Debris Inclination is 88.50677459629775
Payload Inclination is 63.22079596453485


As can be seen from the information above, the eccentricities, balistic coefficients, and inclinations between debris and payload classes differ significantly. Thus, it intuitively makes sense that the random forest classifier chose those three parameters as the top features to split over. It is important to note that if you run the code multiple times, you will see that the balistic coefficient and inclination switch places in terms of importance, but eccentricity remains as the top feature. This makes sense since, on average, the satellites will maintain their orbits and energy through their trusters, whereas over time, the debris will lose energy, resulting in a more eccentric orbit, making the eccentricity the logical top predictor of the object types.